# Phase 2: Illumination-Robust Preprocessing
### Simple, Step-by-Step Interactive Guide & Visual Walkthrough
---
### What is Phase 2 and why do we need it?
Real-world smartphone photos of handwritten documents almost always suffer from:
1. **Uneven Lighting & Shadows:** Room lights, hand/palm shadows, and camera angle gradients make parts of the paper dark and other parts bright.
2. **Paper Creases & Yellowing:** Paper is rarely pure flat white; it has texture, creases, and tint.
3. **Faint Ink Strokes:** Low-contrast ballpoint pen or pencil writing can get lost under simple thresholding.

If we just use a basic threshold (like global Otsu), half the page turns completely black due to shadows!

**Phase 2 solves this in 5 simple steps:**
- **Step 1 (Grayscale):** Convert the 3-channel color photo to single-channel brightness.
- **Step 2 (Background Lighting Estimation):** Use morphological closing to capture the shadow envelope across the paper surface.
- **Step 3 (Lighting Division):** Divide the image by the background lighting to cancel out shadows and make the paper evenly white.
- **Step 4 (Contrast Boost via CLAHE):** Locally enhance the contrast between pen ink and paper.
- **Step 5 (Sauvola Adaptive Binarization):** Convert to a clean inverted binary image (white ink = 255, black paper = 0) ready for character segmentation.

In [ ]:
# Step 1: Import required libraries and Phase 2 modules
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add Model folder to Python path
module_path = Path("../Model").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

from ingestion import load_image, ingest_image
from preprocessing import (
    convert_to_grayscale,
    estimate_background,
    normalize_illumination,
    enhance_contrast_clahe,
    binarize_sauvola,
    binarize_gaussian_adaptive,
    despeckle_mask,
    preprocess_image,
    PreprocessingEngine,
)
import config

print("All Phase 2 modules loaded successfully!")

---
## Step 2: Load Ingested Document from Phase 1
Let us take one of our test images from `data/samples/`.

In [ ]:
# Load test sample
sample_file = Path("../data/samples/IMG_20261021_212517.jpg")
if not sample_file.exists():
    sample_file = Path("../data/samples/sample_camera_capture_synth.png")

# Ingest through Phase 1
original_image, ingested_image, meta = ingest_image(sample_file)
print(f"Ingested Image Shape: {ingested_image.shape} (H, W, Channels)")

---
## Step 3: Grayscale Conversion
Convert BGR colors to standard luminance: $I_{\text{gray}} = 0.299R + 0.587G + 0.114B$.

In [ ]:
gray = convert_to_grayscale(ingested_image)
print(f"Grayscale Image Shape: {gray.shape} (Single channel, uint8)")

---
## Step 4: Estimating Background Lighting (Morphological Closing)
How do we know which parts of the paper are shadowed?

We use **Morphological Closing** with a large elliptical kernel (`cv2.morphologyEx(gray, cv2.MORPH_CLOSE, kernel)`).
- Thin dark strokes (the handwriting) are completely covered over and smoothed away.
- What remains is an estimate of the **smooth paper lighting and shadow surface** alone!

In [ ]:
bg_estimate = estimate_background(gray, kernel_size=41)
print(f"Background Lighting Estimate Shape: {bg_estimate.shape}")
print(f"Min Lighting: {bg_estimate.min()}, Max Lighting: {bg_estimate.max()}")

---
## Step 5: Background Division & Normalization
To remove shadows, we divide the original grayscale image by the estimated lighting surface:
$$\text{Normalized} = \text{clip}\left( \frac{I_{\text{gray}} + 1.0}{I_{\text{bg}} + 1.0} \times 255.0, \; 0, \; 255 \right)$$

*Why the `+ 1.0`?* That is $\epsilon$ (epsilon)—it guarantees we **never divide by zero** if a pixel is pitch black!

In [ ]:
normalized = normalize_illumination(gray, bg_estimate, epsilon=1.0)
print(f"Normalized Image Shape: {normalized.shape}")
print(f"Contains NaNs: {np.isnan(normalized).any()} | Contains Infs: {np.isinf(normalized).any()}")

---
## Step 6: Local Contrast Enhancement via CLAHE
**CLAHE** (Contrast Limited Adaptive Histogram Equalization) splits the image into small $8 \times 8$ tiles and equalizes each tile independently.

- Standard histogram equalization washes out the entire page.
- CLAHE only boosts local contrast, making fine ink strokes crisp and clear.

In [ ]:
enhanced = enhance_contrast_clahe(normalized, clip_limit=2.0, grid_size=(8, 8))
print(f"CLAHE Enhanced Image Shape: {enhanced.shape}")

---
## Step 7: Sauvola Adaptive Binarization
**Sauvola's algorithm (Sauvola & Pietikainen, 2000)** is the gold-standard classical formula for old/uneven documents.

For every pixel, it calculates:
- Local average brightness $m(x, y)$
- Local standard deviation $s(x, y)$
- Dynamic threshold: $T = m \times [1 + k \times (s / R - 1)]$

Output: **Inverted Binary Mask** where:
- **255 (White):** Ink strokes (Foreground)
- **0 (Black):** Paper surface (Background)

In [ ]:
raw_binary = binarize_sauvola(enhanced, window_size=25, k=0.2, r=128)

# Clean tiny dust speckles using a small median blur
clean_binary = despeckle_mask(raw_binary, method="median", kernel_size=3)

ink_pixels = (clean_binary == 255).sum()
total_pixels = clean_binary.size
print(f"Detected Ink Pixels : {ink_pixels:,} ({ink_pixels / total_pixels * 100:.2f}% of page)")
print(f"Background Pixels   : {total_pixels - ink_pixels:,}")

---
## Step 8: Visualizing the Entire Preprocessing Pipeline
Let us inspect all intermediate stages side-by-side to clearly see how lighting is corrected and ink is isolated.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Ingested Input
axes[0, 0].imshow(cv2.cvtColor(ingested_image, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("1. Ingested Image (Phase 1 Output)", fontsize=11, fontweight="bold")
axes[0, 0].axis("off")

# 2. Grayscale
axes[0, 1].imshow(gray, cmap="gray")
axes[0, 1].set_title("2. Grayscale (Luminance)", fontsize=11, fontweight="bold")
axes[0, 1].axis("off")

# 3. Background Lighting Envelope
axes[0, 2].imshow(bg_estimate, cmap="gray")
axes[0, 2].set_title("3. Background Lighting (Morphological Close)", fontsize=11, fontweight="bold")
axes[0, 2].axis("off")

# 4. Background-Divided Normalized
axes[1, 0].imshow(normalized, cmap="gray")
axes[1, 0].set_title("4. Lighting Normalized (Shadows Removed)", fontsize=11, fontweight="bold")
axes[1, 0].axis("off")

# 5. CLAHE Contrast Boost
axes[1, 1].imshow(enhanced, cmap="gray")
axes[1, 1].set_title("5. CLAHE Enhanced (Crisp Strokes)", fontsize=11, fontweight="bold")
axes[1, 1].axis("off")

# 6. Final Clean Inverted Binary Mask
axes[1, 2].imshow(clean_binary, cmap="gray")
axes[1, 2].set_title("6. Final Inverted Binary Mask (White = Ink)", fontsize=11, fontweight="bold")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

---
## Quick Professor & Viva Cheat Sheet (Phase 2)

> **Q1: Why can't we use simple global Otsu thresholding directly?**  
> *Answer:* Real phone photographs have uneven lighting and shadows. A single global threshold turns shadowed areas completely black and bright areas washed out. Local illumination normalization is mandatory.
>
> **Q2: How does morphological closing estimate the background lighting?**  
> *Answer:* By using an elliptical kernel larger than the handwriting stroke thickness, the closing operation (dilation followed by erosion) bridges across and erases the dark ink strokes, leaving only the smoothly varying illumination surface.
>
> **Q3: What is the purpose of $\epsilon = 1.0$ in the division formula?**  
> *Answer:* It is a smoothing constant that strictly prevents division by zero and numerical instability when dividing pixel intensities.
>
> **Q4: Why Sauvola binarization instead of standard fixed thresholding?**  
> *Answer:* Sauvola uses both the local mean and local standard deviation in a moving window. In low-contrast regions with faint handwriting, the threshold automatically adapts to capture the faint ink without picking up paper noise.
>
> **Q5: Why is the binary mask inverted (white ink, black background)?**  
> *Answer:* In classical image processing and contour extraction (OpenCV), foreground objects are conventionally represented by non-zero (255) pixels and background as zero (0).